In [5]:
import dataclasses
import time

import flax
import jax
import orbax
from flax.training import orbax_utils
from jax.tree_util import Partial

from crew.experiments.paths import build_trained_weights_path
from crew.RND.config import TrainConfig
from crew.RND.main_loop import full_training_on_fixed_envs
from crew.RND.setups import set_up_for_training
from crew.shared_code.logging import (
    generate_run_name,
    wandb_log_training_metrics,
)


In [ ]:
# check gpu usage
import jax

print(jax.default_backend())
print(jax.devices())

import jax.numpy as jnp

x = jnp.ones((1000, 1000))
print(x.device)

y = x @ x
y.block_until_ready()
print(y.device)


gpu
[CudaDevice(id=0)]
cuda:0
cuda:0


In [6]:
def run_rnd_training(config: TrainConfig, save_results: bool = True, log_to_wandb: bool = True):
    # setup
    rng, env, env_params, agent_train_state, predictor_train_state, target_train_state = set_up_for_training(config)

    # train
    print(f"Training with seed {config.train_seed}")
    t = time.time()

    full_training_partial = Partial(full_training_on_fixed_envs, env=env, env_params=env_params, config=config)
    jitted_full_training = jax.jit(full_training_partial)
    train_info = jax.block_until_ready(jitted_full_training(rng=rng, agent_train_state=agent_train_state, predictor_train_state=predictor_train_state, target_train_state=target_train_state))

    elapsed_time = time.time() - t
    print(f"Done in {elapsed_time / 60:.2f}min")

    if save_results:
        try:
            # store results on disk
            save_path = build_trained_weights_path(
                algorithm_id="rnd",
                env_id=config.env_id,
                seed=config.train_seed,
                artifacts_root=config.artifacts_root,
            )
            # timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            # save_path = save_path.parent / f"{save_path.name}_{timestamp}"
            save_path.parent.mkdir(parents=True, exist_ok=True)

            train_config = dataclasses.asdict(config)
            train_config = flax.core.freeze(train_config)
            training_results = {
                "config": train_config,
                "agent_params": train_info["agent_state"].params,
                "predictor_params": train_info["predictor_state"].params,
                "metrics": train_info["metrics"],
            }

            orbax_checkpointer = orbax.checkpoint.PyTreeCheckpointer()
            save_args = orbax_utils.save_args_from_target(training_results)
            orbax_checkpointer.save(save_path, training_results, save_args=save_args)
            print("saved training results to", save_path)
        except Exception as e:
            print(f"Error while saving training results to disk: {e}")

    if log_to_wandb:
        # save logs to wandb
        extra_logs = {}
        if "lr" in train_info["metrics"]:
            extra_logs["training/lr"] = train_info["metrics"]["lr"]
        if "rnd_predictor_loss" in train_info["metrics"]:
            extra_logs["predictor/loss"] = train_info["metrics"]["rnd_predictor_loss"]
        run_name = generate_run_name(algorithm_name="RND", config=config, prefix="")
        tags = ["rnd", "train"]
        wandb_log_training_metrics(train_info["metrics"], config, run_name, project_name="open_endedness", tags=tags, extra_batch_metrics=extra_logs)
        time.sleep(15)


In [ ]:
train_seeds = [10, 20, 30, 40]
total_timesteps = 1_000_000_000
env_id = "Craftax-Classic-Symbolic-v1"

for seed in train_seeds:
    config = TrainConfig(
        train_seed=seed,
        total_timesteps=total_timesteps,
        env_id=env_id,
    )

    run_rnd_training(config)


In [ ]:
# sanity test to run locally
jax.config.update("jax_log_compiles", True)


config = TrainConfig(
    train_seed=1,
    total_timesteps=50_000,
    env_id="Craftax-Classic-Symbolic-v1",
    num_envs_per_batch=64,
    num_steps_per_env=512,
    num_steps_per_update=256,
    update_epochs=1,
    num_minibatches=16,
    #
    adam_eps=1e-5,
    lr=2e-4,
    anneal_lr=False,
    clip_eps=0.2,
    gamma=0.99,
    gae_lambda=0.95,
    ent_coef=0.005,
    vf_coef=0.5,
    max_grad_norm=0.5,
    # encoders
    obs_emb_dim=256,
    # Transformer XL specific
    past_context_length=64,
    subsequence_length_in_loss_calculation=64,
    num_attn_heads=4,
    num_transformer_blocks=1,
    transformer_hidden_states_dim=64,
    qkv_features=64,
    gating=True,
    gating_bias=2.0,
    # actor and critic head MLPs
    head_activation="relu",
    head_hidden_dim=64,
    # RND specific
    rnd_encoder_mode="flat_symbolic",
    rnd_output_embedding_dim=64,  # 64
    rnd_head_activation="relu",
    rnd_head_hidden_dim=64,
    eval_num_envs=32,
)
run_rnd_training(config=config, save_results=False, log_to_wandb=False)

Loading Craftax-Classic textures from cache.
Textures successfully loaded from cache.
Training with seed 1


E0219 20:53:34.969643  649759 slow_operation_alarm.cc:73] 
********************************
[Compiling module jit__unknown for GPU] Very slow compile? If you want to file a bug, run with envvar XLA_FLAGS=--xla_dump_to=/tmp/foo and attach the results.
********************************
